In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
query = "What is expected credit loss?"

query_embedding = embedding_model.embed_query(query)

In [9]:
type(query_embedding)

list

In [10]:
len(query_embedding)

384

In [11]:
query_embedding[:10]

[-0.011116755194962025,
 0.06500456482172012,
 -0.0008477629744447768,
 0.005995448213070631,
 0.07649991661310196,
 0.05195862799882889,
 0.04387192055583,
 0.04969889298081398,
 0.05209346115589142,
 0.053306594491004944]

In [12]:
sentences = [
    "A financial asset moves to Stage 2 when credit risk increases significantly.",
    
    "Lifetime expected credit losses are recognised when there is a significant increase in credit risk.",
    
    "The weather in Delhi is extremely hot today."
]

In [13]:
sentence_embeddings = embedding_model.embed_documents(sentences)

In [14]:
len(sentence_embeddings)

3

In [16]:
len(sentence_embeddings[0])

384

In [19]:
import numpy as np

In [21]:
vectors = np.array(sentence_embeddings)

vectors.shape

(3, 384)

In [22]:
similarity_1_2 = np.dot(
    vectors[0],
    vectors[1]
)

similarity_1_3 = np.dot(
    vectors[0],
    vectors[2]
)

print(
    "Credit Risk sentence vs Credit Risk sentence:",
    similarity_1_2
)

print(
    "Credit Risk sentence vs Weather sentence:",
    similarity_1_3
)

Credit Risk sentence vs Credit Risk sentence: 0.4836221702498502
Credit Risk sentence vs Weather sentence: -0.05673693099655441


In [25]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: d:\AI Projects\Credit-Risk-Knowledge-Assistant


In [26]:
from app.services.document_loader import (
    DOCUMENT_DIR,
    load_all_pdfs
)

from app.services.text_splitter import (
    split_documents
)

In [27]:
documents = load_all_pdfs(
    DOCUMENT_DIR
)

chunks = split_documents(
    documents,
    chunk_size=1000,
    chunk_overlap=200
)

print("Documents:", len(documents))
print("Chunks:", len(chunks))


Found 3 PDF file(s).

Loading: INDAS109.pdf
Loaded 189 page(s)

Loading: Master Circular - Bank Finance to Non-Banking Financial Companies (NBFCs).pdf
Loaded 11 page(s)

Loading: Master Circular - Prudential norms on Income Recognition, Asset Classification and.pdf
Loaded 77 page(s)

Total pages/documents loaded: 277
Documents: 277
Chunks: 880


In [28]:
sample_chunks = chunks[:10]

In [30]:
sample_texts = [
    chunk.page_content
    for chunk in sample_chunks
]


sample_embeddings = (
    embedding_model.embed_documents(
        sample_texts
    )
)

In [31]:
len(sample_embeddings)

10

In [32]:
sample_chunks = chunks[:100]

In [33]:
sample_texts = [
    chunk.page_content
    for chunk in sample_chunks
]

sample_embeddings = (
    embedding_model.embed_documents(
        sample_texts
    )
)

document_vectors = np.array(
    sample_embeddings
)

In [34]:
question = (
    "What is the objective of Ind AS 109?"
)

In [35]:
question_vector = np.array(
    embedding_model.embed_query(
        question
    )
)

In [36]:
question_vector = np.array(
    embedding_model.embed_query(
        question
    )
)

In [37]:
similarities = (
    document_vectors @ question_vector
)

In [38]:
top_indices = np.argsort(
    similarities
)[::-1][:5]

In [39]:
for rank, index in enumerate(
    top_indices,
    start=1
):
    
    chunk = sample_chunks[index]
    
    print("=" * 80)
    
    print(
        f"Rank: {rank}"
    )
    
    print(
        f"Similarity: "
        f"{similarities[index]:.4f}"
    )
    
    print(
        "Source:",
        chunk.metadata.get(
            "source_file"
        )
    )
    
    print(
        "Page:",
        chunk.metadata.get(
            "page_number"
        )
    )
    
    print()
    
    print(
        chunk.page_content[:700]
    )
    
    print()

Rank: 1
Similarity: 0.4678
Source: INDAS109.pdf
Page: 31

contingent consideration recognised by an acquirer in a business 
combination to which Ind AS 103 applies. (See paragrap h B5.7.3 
for guidance on foreign exchange gains or losses.)  
 
5.7.6 If an entity makes the election in paragraph 5.7.5, it shall recognise in 
profit or loss dividends from that investment in accordance with 
paragraph 5.7.1A.  
 
Liabilities designated as at fair value through profit or loss 
 
5.7.7 An entity shall present a gain or loss on a financial liability that is 
designated as at fair value through profit or loss in accordance 
with paragraph 4.2.2 or paragraph 4.3.5 as follows:

Rank: 2
Similarity: 0.4556
Source: INDAS109.pdf
Page: 1

246 
 
 
Indian Accounting Standard (Ind AS) 109 
Financial Instruments 
 
(The Indian Accounting Standard includes paragraphs set in bold type and plain 
type, which have equal authority. Paragraphs in bold type indicate the main 
principles.) 
 
 
Chapter 1 Object